In [37]:
import os
import traceback
import pdb #debug
import time
import shutil
import math
import re
import csv
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

import networkx as nx
#import pydot #deprecated
import pygraphviz as pgv
from networkx.drawing.nx_agraph import read_dot, from_agraph
from graphviz import Digraph
from graphviz import Source

#Neurmorphology and Simulation
import neuroarch.na as na


from IPython.display import display, JSON
import json
from scipy import stats

# ANSI color codes
GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"

In [36]:
#turn on breakpoints?
%pdb on

Automatic pdb calling has been turned ON


In [4]:
!git init
!git remote add origin https://github.com/sgarnell/archive

Reinitialized existing Git repository in /home/ffbo/ffbo/.git/
fatal: remote origin already exists.


In [4]:
!git remote -v

origin	https://github.com/sgarnell/archive.git (fetch)
origin	https://github.com/sgarnell/archive.git (push)


In [95]:
# Stage the notebook file
!git add 'PC_Latex_Generator_v1.ipynb'

# Commit the changes
!git commit -m "Auto commit from Jupyter for {notebook_name}"

# Push to GitHub
!git push origin fbl2-main  # Replace 'main' with 'master' if needed

[fbl2-main 619d0e1] Auto commit from Jupyter for {notebook_name}
 1 file changed, 1839 insertions(+), 137 deletions(-)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 16 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 12.61 KiB | 6.30 MiB/s, done.
Total 3 (delta 1), reused 0 (delta 0)
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/sgarnell/archive.git
   e0910cf..619d0e1  fbl2-main -> fbl2-main


In [133]:
import os
import json
import re

#Load data from 3 files provided
#  - Dot file with graphviz graph
#  - nt_file contains the neurotransmitter data for each neuron
#  - Motif contains data about specific neuron combinations, ie Wilson-Cowan
def load_files(dot_file, nt_file, motif_file):
    print("Getting data from files:")
    with open(dot_file, "r") as f:
        dot_lines = f.readlines()
    with open(nt_file, "r") as f:
        nt_dict = json.load(f)
    with open(motif_file, "r") as f:
        motif_dict = json.load(f)
        
    return dot_lines, nt_dict, motif_dict


def get_polarity(neuron, nt_dict, polarity_dict):
    nts = nt_dict.get(neuron, {})
    for nt in nts:
        sign = polarity_dict.get(nt.lower())
        if sign is not None:
            return "+" if sign == 1 else "-"
    return "?"

def extract_prefix(name, cutoff):
    return name[:cutoff]


def extract_edges(dot_lines):
    edges = []
    for line in dot_lines:
        match = re.search(
            r'"?([^"\s]+)"? -> "?([^"\s]+)"? \[label="Zscore = ([0-9.]+)"', line
        )
        if match:
            head, tail, zscore = match.groups()
            edges.append((head, tail, float(zscore)))
        elif "->" in line:
            print("Skipped edge-like line:", line)
    return edges



def get_synapse_name(head, tail):
    if "--" in head:
        return head
    if "--" in tail:
        return tail
    return None

def extract_type_and_region(neuron):
    match = re.match(r"([A-Za-z0-9_]+)\(([^)]+)\)(\d*)", neuron)
    if match:
        typ, region, index = match.groups()
        return typ, region
    return neuron, None


def abstract_synapse_name(synapse, cutoff):
    if "--" not in synapse:
        return synapse
    source, target = synapse.split("--")
    src_type, src_region = extract_type_and_region(source)
    tgt_type, tgt_region = extract_type_and_region(target)

    # Fallback to cutoff if region parsing fails
    if not src_type or not tgt_type:
        return f"{source[:cutoff]}-{target[:cutoff]}"

    return f"{src_type}-{tgt_type}"


def get_wc_neurons(motif_dict):
    wc_neurons = set()
    if motif_dict:
        for motif in motif_dict.values():
            if abbreviate_motif_type(motif.get("motifType", "unknown")).upper() == "WC":
                wc_neurons.add(motif["neuron_A"])
                wc_neurons.add(motif["neuron_B"])
    return wc_neurons

def find_best_parallel_cutoff(edges, min_cutoff=3, max_cutoff=20, exclude_neurons=None):
    if exclude_neurons is None:
        exclude_neurons = set()

    cutoff_scores = {}
    cutoff_groups = {}

    for cutoff in range(min_cutoff, max_cutoff + 1):
        groups = defaultdict(list)

        for head, tail, z in edges:
            # Identify the neuron leg (not the synapse)
            neuron = tail if "--" in head else head
            if neuron in exclude_neurons:
                continue

            prefix = extract_prefix(neuron, cutoff)
            groups[prefix].append((head, tail, z))

        valid_groups = [g for g in groups.values() if len(g) > 1]
        group_count = len(valid_groups)
        total_members = sum(len(g) for g in valid_groups)
        avg_group_size = total_members / group_count if group_count > 0 else 0

        cutoff_scores[cutoff] = {
            "group_count": group_count,
            "total_members": total_members,
            "avg_group_size": avg_group_size
        }
        cutoff_groups[cutoff] = groups

    # Step 2: Detect plateau in avg_group_size
    plateau_cutoffs = []
    for cutoff in range(min_cutoff + 1, max_cutoff):
        prev_avg = cutoff_scores[cutoff - 1]["avg_group_size"]
        curr_avg = cutoff_scores[cutoff]["avg_group_size"]
        next_avg = cutoff_scores[cutoff + 1]["avg_group_size"]
        if abs(curr_avg - prev_avg) < 1.0 and abs(curr_avg - next_avg) < 1.0:
            plateau_cutoffs.append(cutoff)

    # Step 3: Choose smallest cutoff in plateau, fallback to max total_members
    if plateau_cutoffs:
        best_cutoff = min(plateau_cutoffs)
    else:
        best_cutoff = max(cutoff_scores, key=lambda k: cutoff_scores[k]["total_members"])

    best_groups = cutoff_groups[best_cutoff]
    best_prefixes = list(best_groups.keys())

    return best_prefixes, best_cutoff



def exclude_motif_edges(edges, motifs_dict):
    motif_neurons = set(motifs_dict.keys())
    filtered = []
    for head, tail, z in edges:
        synapse = get_synapse_name(head, tail)
        reverse_synapse = None
        if synapse and "--" in synapse:
            a, b = synapse.split("--")
            reverse_synapse = f"{b}--{a}"

        if (
            head in motif_neurons or
            tail in motif_neurons or
            synapse in motifs_dict or
            reverse_synapse in motifs_dict
        ):
            continue

        filtered.append((head, tail, z))
    return filtered


def latex_clean(name):
    return (
        name.replace("_", "")
            .replace("--", "-")
            .replace("(", "")
            .replace(")", "")
            .replace(">", "to")
    )

def identify_dyad_roles(group, nt_dict, polarity_dict):
    if len(group) != 2:
        return None

    def is_synapse(name):
        return '--' in name

    def parse_synapse(name):
        parts = name.split('--', 1)
        if len(parts) != 2:
            raise ValueError(f"Malformed synapse name: {name}")
        return latex_clean(parts[0]), latex_clean(parts[1])

    # Identify which leg contains the synapse
    syn_leg = None
    other_leg = None
    for leg in group:
        h, t, z = leg
        if is_synapse(h):
            syn_leg = ('head', h, t, z)
        elif is_synapse(t):
            syn_leg = ('tail', h, t, z)
        else:
            raise ValueError(f"Invalid dyad: no synapse found in leg {leg}")

    # Assign the other leg
    other_leg = [leg for leg in group if leg != syn_leg[1:]][0]
    h_other, t_other, z_other = other_leg

    # Parse synapse and determine directionality
    syn_loc, h_syn, t_syn, z_syn = syn_leg
    syn_raw = h_syn if syn_loc == 'head' else t_syn
    synapse = latex_clean(syn_raw)
    syn_pre, syn_post = parse_synapse(syn_raw)

    if syn_loc == 'head':
        if latex_clean(t_syn) == syn_post:
            pre = latex_clean(h_other)
            post = syn_post
            z_pre, z_post = z_other, z_syn
        else:
            raise ValueError(f"Tail of synapse leg does not match postsynaptic neuron: {t_syn}")
    else:  # synapse in tail
        if latex_clean(h_syn) == syn_pre:
            pre = syn_pre
            post = latex_clean(t_other)
            z_pre, z_post = z_syn, z_other
        else:
            raise ValueError(f"Head of synapse leg does not match presynaptic neuron: {h_syn}")

    # Determine polarity
    polarity1 = get_polarity(h_syn, nt_dict, polarity_dict)
    polarity2 = get_polarity(h_other, nt_dict, polarity_dict)
    polarity = polarity1 if polarity1 == polarity2 else f"{polarity1}/{polarity2}"

    return {
        "pre": latex_clean(pre),
        "pre_raw": pre,
        "post": latex_clean(post),
        "post_raw": post,
        "synapse": latex_clean(syn_raw),
        "synapse_raw": syn_raw,
        "z_expr": f"z^{{{z_post:.2f}/{z_pre:.2f}}}",
        "polarity": polarity
    }





def identify_ensemble_roles(group, nt_dict, polarity_dict):
    if len(group) < 2:
        return None

    def is_synapse(name):
        return '--' in name

    def parse_synapse(name):
        parts = name.split('--', 1)
        if len(parts) != 2:
            raise ValueError(f"Malformed synapse name: {name}")
        return latex_clean(parts[0]), latex_clean(parts[1])

    # Step 1: Identify all legs with a synapse
    syn_legs = []
    other_legs = []
    for h, t, z in group:
        if is_synapse(h):
            syn_legs.append(('head', h, t, z))
        elif is_synapse(t):
            syn_legs.append(('tail', h, t, z))
        else:
            other_legs.append((h, t, z))

    if not syn_legs:
        return None

    # Step 2: Extract synapse names and pick the most common
    syn_raws = [h if loc == 'head' else t for loc, h, t, z in syn_legs]
    syn_raw = Counter(syn_raws).most_common(1)[0][0]
    synapse = latex_clean(syn_raw)
    syn_pre, syn_post = parse_synapse(syn_raw)

    # Step 3: Collect pre/post candidates and z-scores
    pre_candidates = []
    post_candidates = []
    z_scores = []
    polarities = set()

    for loc, h_syn, t_syn, z_syn in syn_legs:
        if h_syn == syn_raw or t_syn == syn_raw:
            if loc == 'head':
                if latex_clean(t_syn) == syn_post:
                    pre_candidates.append(latex_clean(h_syn))
                    post_candidates.append(syn_post)
                    z_scores.append(z_syn)
                    polarities.add(get_polarity(h_syn, nt_dict, polarity_dict))
            else:  # synapse in tail
                if latex_clean(h_syn) == syn_pre:
                    pre_candidates.append(syn_pre)
                    post_candidates.append(latex_clean(t_syn))
                    z_scores.append(z_syn)
                    polarities.add(get_polarity(h_syn, nt_dict, polarity_dict))

    if not pre_candidates or not post_candidates:
        return None

    pre_raw = pre_candidates[0]
    post_raw = post_candidates[0]

    polarity = polarities.pop() if len(polarities) == 1 else "/".join(sorted(polarities))
    avg_z = sum(z_scores) / len(z_scores)

    return {
        "pre": latex_clean(pre_raw),
        "pre_raw": pre_raw,
        "post": latex_clean(post_raw),
        "post_raw": post_raw,
        "synapse": synapse,
        "synapse_raw": syn_raw,
        "z_expr": f"z^{{{avg_z:.2f}}}",
        "polarity": polarity
    }





def build_neuron_equations(edges, nt_dict, polarity_dict, motif_dict=None):
    
    recurrent_count = 0
    dyad_count = 0
    ensemble_count = 0
    fanin_count = 0
    fanout_count = 0
    microstate_count = 0

    
    used_heads = set()
    used_tails = set()
    used_synapses = set()
    
    #remove all known Motifs from the edge data
    xmotif_edges = exclude_motif_edges(edges, motif_dict)
    equations = []
    wc_neurons = get_wc_neurons(motif_dict)
    


    # Step 1: Build recurrent motif equations
    recurrent_dict = {}
    recurrent_groups = defaultdict(list)
    edge_set = set((head, tail) for head, tail, _ in edges)

    # Step 1a: Get candidate recurrent neuron prefixes
    recurrent_prefixes, cutoff = find_best_parallel_cutoff(edges, exclude_neurons=wc_neurons)

    # Step 1b: Group edges by recurrent synapse identity based on string splitting
    def get_prefix(name):
        for delimiter in ["_", "(", "-"]:
            if delimiter in name:
                return name.split(delimiter, 1)[0]
        return name


    for head, tail, z in edges:
        synapse = head if "--" in head else tail
        pre, post = synapse.split("--")

        pre_prefix = get_prefix(pre)
        post_prefix = get_prefix(post)


        # Build recurrent list based on matching pre and post
        if pre_prefix == post_prefix:
            recurrent_groups[f"{pre_prefix}--{post_prefix}"].append((head, tail, z))
            used_synapses.add(synapse)
            #print(f"Matching pre {pre_prefix} and post {post_prefix}")

    ############################
    # Step 1c: Emit equations
    for synapse, group in recurrent_groups.items():
        if len(group) < 2:
            continue  # skip trivial motifs

        pre, post = synapse.split("--")
        motif_label = f"\\mathcal{{R}}_{{{latex_clean(pre)} \\leftrightarrow {latex_clean(post)}}}"

        eq = (
            "\\begin{align*}\n"
            f"{motif_label} = \\sum_{{i,j}} "
            f"\\bm{{W}}_{{{latex_clean(pre)} \\leftrightarrow {latex_clean(post)}}}^{{(i,j)}} "
            f"\\cdot \\mu_{{{latex_clean(pre)}, {latex_clean(post)}}}^{{(i,j)}}\n"
            "\\end{align*}"
        )

        source_neurons_list = list(set(latex_clean(h) for h, _, _ in group))
        target_neurons_list = list(set(latex_clean(t) for _, t, _ in group))

        equations.append({
            "label": motif_label,
            "type": "recurrent",
            "latex": eq,
            "group_size": len(group),
            "source_neurons": source_neurons_list,
            "target_neurons": target_neurons_list,
            "edges": [(head, tail, z) for head, tail, z in group]
        })


        for head, tail, _ in group:
            if '--' not in head:
                used_heads.add(head)
            else:
                used_synapses.add(head)
                used_tails.add(tail)

        recurrent_count += 1
        recurrent_dict[synapse] = group

    print(f"{len(recurrent_dict)} recurrent motifs found")
    display(JSON(recurrent_dict))


    #########################    
    #########################
    #Step 2 Fan-In Motif logic
    fan_in_map = defaultdict(list)
    fanin_dict = {}
    for head, tail, z in xmotif_edges:
        
        synapse = get_synapse_name(head, tail)
        
        #skp this routine if WC types
        if tail in wc_neurons or synapse in used_synapses:
            continue
        if "--" in tail: #skip all tails that are synapses
            continue
        
            
        fan_in_map[tail].append((head, z))
        
    fan_in_cutoff = 3 #assume that the first 3 signify a group prefix here

    #Step 2.a Prefix map that we use to group strong candidates
    #All heads are now treated as synapses, they must be for this to work
    for target, incoming_legs in fan_in_map.items():
        prefix_map = defaultdict(list)
        for synapse, z in incoming_legs:
            pre, post = synapse.split("--")
            prefix = extract_prefix(pre,fan_in_cutoff)
            prefix_map[prefix].append((synapse, z))

    
        strong_prefixes = {p: legs for p, legs in prefix_map.items() if len(legs) >= 3}
    
        if not strong_prefixes:
            continue  # No strong fan-in motifs


        #Step 2.b
        for prefixSource , incoming_legs in strong_prefixes.items():
            

            #Fanin total Count
            fanin_count += 1
            ensemble_label = f"\\mu^\\epsilon_{{{target}}}^{{(P)}}"

            terms = []
            source_neurons = []
            edge_list = []
            sources = set()

            for source, z in incoming_legs:
                used_synapses.add(source)
                pre, post = source.split("--")
                polarity = get_polarity(pre, nt_dict, polarity_dict)
                term = (
                    f"\\bm{{W}}_{{{latex_clean(target)} \\leftarrow {latex_clean(pre)}}}^{{({polarity})}}(z^{{{z:.2f}}}) "
                    f"\\cdot \\mu_{{{latex_clean(pre)}}}^{{(A)}}"
                )

                terms.append(term)
                source_neurons.append(pre)
                edge_list.append((pre, post, z))
                sources.add(latex_clean(pre))

            eq = (
                "\\begin{align*}\n"
                f"{ensemble_label} = & " + " \\\\\n+ & ".join(terms) + "\n"
                "\\end{align*}"
            )

            equations.append({
                "label": ensemble_label,
                "type": "fan_in",
                "latex": eq,
                "group_size": len(sources),
                "source_neurons": source_neurons,
                "target_neurons": [post],
                "edges": edge_list
            })


            fanin_dict[post] = source_neurons
    
    print(f"{len(fanin_dict)} Fan-in neurons found")
    display(JSON(fanin_dict))

    
    #########################    
    #########################
    ##########################
    #Step 3 Fan-OUt Motif logic
    fan_out_map = defaultdict(list)
    fanout_dict = {}
    for head, tail, z in xmotif_edges:
        
        synapse = get_synapse_name(head, tail)
        
        #skp this routine if WC types
        if head in wc_neurons or synapse in used_synapses:
            continue
        if "--" in head: #skip all heads that are synapses
            continue
        
            
        fan_out_map[head].append((tail, z))
        
    fan_out_cutoff = 3 #assume that the first 3 signify a group prefix here

    
    #Step 3.a Prefix map that we use to group strong candidates
    #All heads are now treated as synapses, they must be for this to work
    for source, outgoing_legs in fan_out_map.items():
        prefix_map = defaultdict(list)
        for synapse, z in outgoing_legs:
            pre, post = synapse.split("--")
            prefix = extract_prefix(post,fan_out_cutoff)
            prefix_map[prefix].append((synapse, z))

    
        strong_prefixes = {p: legs for p, legs in prefix_map.items() if len(legs) >= 3}
    
        if not strong_prefixes:
            continue  # No strong fan-in motifs


        #Step 3.b
        for prefixTarget , outgoing_legs in strong_prefixes.items():
            

            #Fanout total Count
            fanout_count += 1
            ensemble_label = f"\\mu^\\epsilon_{{{source}}}^{{(P)}}"

            terms = []
            target_neurons = []
            edge_list = []
            targets = set()

            for target, z in outgoing_legs:
                used_synapses.add(target)
                pre, post = target.split("--")
                polarity = get_polarity(post, nt_dict, polarity_dict)
                term = (
                    f"\\bm{{W}}_{{{latex_clean(source)} \\leftarrow {latex_clean(post)}}}^{{({polarity})}}(z^{{{z:.2f}}}) "
                    f"\\cdot \\mu_{{{latex_clean(post)}}}^{{(A)}}"
                )

                terms.append(term)
                target_neurons.append(post)
                edge_list.append((pre, post, z))
                targets.add(latex_clean(post))

            eq = (
                "\\begin{align*}\n"
                f"{ensemble_label} = & " + " \\\\\n+ & ".join(terms) + "\n"
                "\\end{align*}"
            )

            equations.append({
                "label": ensemble_label,
                "type": "fan_out",
                "latex": eq,
                "group_size": len(targets),
                "source_neurons": [pre],
                "target_neurons": target_neurons,
                "edges": edge_list
            })


            fanout_dict[pre] = target_neurons
    
    print(f"{len(fanout_dict)} Fan-out neurons found")
    display(JSON(fanout_dict))
    
    # Step 3: Fan-Out Motif logic (one source → many targets)
#     fanout_dict = {}
#     fan_out_map = defaultdict(list)
#     for head, tail, z in xmotif_edges:
        
        
        
#         if head in wc_neurons or head in used_heads:
#             continue
#         if "--" in head:
#             continue  # skip compound synapses

#         fanout_count += 1
#         fan_out_map[head].append((tail, z))  # raw tail preserved for tracking
        
    

#     for source, outgoing_legs in fan_out_map.items():
#         if len(outgoing_legs) < 2:
#             continue  # not a true fan-out

#         ensemble_label = f"\\mu^\\epsilon_{{{latex_clean(source)}}}^{{(A)}}"
#         terms = []
#         target_neurons = []
#         edge_list = []
#         targets = set()

#         for target, z in outgoing_legs:
#             polarity = get_polarity(source, nt_dict, polarity_dict)

#             term = (
#                 f"\\bm{{W}}_{{{latex_clean(target)} \\leftarrow {latex_clean(source)}}}^{{({polarity})}}(z^{{{z:.2f}}}) "
#                 f"\\cdot \\mu_{{{latex_clean(source)}}}^{{(A)}}"
#             )

#             terms.append(term)
#             target_neurons.append(target)
#             edge_list.append((source, target, z))
#             targets.add(latex_clean(target))

#             eq = (
#                 "\\begin{align*}\n"
#                 f"{ensemble_label} = & " + " \\\\\n+ & ".join(terms) + "\n"
#                 "\\end{align*}"
#             )

#             equations.append({
#                 "label": ensemble_label,
#                 "type": "fan_out",
#                 "latex": eq,
#                 "group_size": len(targets),
#                 "source_neurons": [source],
#                 "target_neurons": target_neurons,
#                 "edges": edge_list
#             })

#             #store neurons for review
#             fanout_dict[source] = target_neurons

#             used_heads.update([source])
#             used_tails.update(target_neurons)
        
#     print(f"\n{len(fanout_dict)} Fan-out neurons found")
#     display(JSON(fanout_dict)) 


    #########################
    #########################
    #########################    
    #########################    
   # Step 4: Ensemble Synapseabstraction using cutoff (2 legged and larger ensembles)
    parallel_groups, cutoff_used = find_best_parallel_cutoff(edges, exclude_neurons=wc_neurons)

    # Build abstracted synapse groups using cutoff
    abstracted_groups = defaultdict(list)
    for head, tail, z in xmotif_edges:
        head = (head)
        tail = (tail)
        synapse = get_synapse_name(head, tail)
        if synapse in used_synapses:
            continue  # skip edges already used in recurrent motifs
        
        if synapse:
            abstracted_key = abstract_synapse_name(synapse, cutoff_used)
            abstracted_groups[abstracted_key].append((head, tail, z))


    # Telemetry store of all roles product for dyads
    dyad_roles_dict = {}
    
    # Emit Dyad ensemble equations
    for key, group in abstracted_groups.items():
        
        group_size = len(group)
        if group_size < 2 or group_size == 3:
            continue  # skip orphaned or ambiguous triads


        
        #DYADS are genearted here
        if group_size == 2:
           
            roles = identify_dyad_roles(group, nt_dict, polarity_dict)
            
            if roles:
                
                #keep track of roles for debug
                dyad_roles_dict[key] = roles  # key is the group identifier
                
                # if any("EPG" in v for v in [roles["pre"], roles["post"], roles["synapse_raw"]]):
                #     print(f"🚨 EPG found in dyad roles: {roles}")
                
                dyad_count += 1 #count the enesmebles of 2 legs
                
                # Simplified dyad form: predictive state of postsynaptic neuron
                label = f"\\mu_{{{roles['post']}}}^{{(P)}}"
                term = (
                    f"\\bm{{W}}_{{{roles['synapse_raw']}}}^{{({roles['polarity']})}}({roles['z_expr']}) "
                    f"\\cdot \\mu_{{{roles['pre']}}}^{{(A)}}"
                )
                
                # # Trap dyads where head is CRE042
                # if "CRE042" in roles['post'] or "CRE042" in roles['pre'] :
                #     print("\n🚨 DEBUG: CRE042 head detected 🚨")
                #     print(f"\\bm{{W}}_{{{roles['synapse_raw']}}}^{{({roles['polarity']})}}({roles['z_expr']}) ")
                #     break  # Only need to print once per group 
                
                eq = (
                    f"% Cutoff used: {cutoff_used}\n"
                    "\\begin{align*}\n"
                    f"{label} = & {term}\n"
                    "\\end{align*}"
                )

                used_heads.update([roles['pre_raw']])
                used_tails.update([roles['post_raw']])

                equations.append({
                    "label": label,
                    "type": "dyad",
                    "latex": eq,
                    "group_size": group_size,
                    "source_neurons": [roles['pre']],
                    "target_neurons": [roles['post']],
                    "edges": [(head, tail, z) for head, tail, z in group]
                })

            else:
                # Fallback: 2 leggs but no roles? Sholdn't exist but that scenario runs here

                head1, tail1, z1 = group[0]
                head2, tail2, z2 = group[1]
                
                print (f"Fallback routine error: 2 legs but no roles for {head1} and {tail1} ")

                polarity1 = get_polarity(head1, nt_dict, polarity_dict)
                polarity2 = get_polarity(head2, nt_dict, polarity_dict)

                synapse_label = latex_clean(key)
                z_expr = f"z^{{{z1:.2f}/{z2:.2f}}}" if abs(z1 - z2) > 0.01 else f"z^{{{(z1 + z2)/2:.2f}}}"

                source_expr = (
                    f"\\mu_{{{latex_clean(head1)}}}^{{(A)}} + \\mu_{{{latex_clean(head2)}}}^{{(A)}}"
                    if latex_clean(head1) != latex_clean(head2)
                    else f"\\mu_{{{latex_clean(head1)}}}^{{(A)}}"
                )

                polarity_expr = polarity1 if polarity1 == polarity2 else f"{polarity1}/{polarity2}"

                termUnifiedLegs = (
                    f"\\bm{{W}}_{{{synapse_label}}}^{{({polarity_expr})}}({z_expr}) "
                    f"\\cdot \\left( {source_expr} \\right)"
                )

                label = f"\\mu_{{{synapse_label}}}^{{(P)}}"
                eq = (
                    f"% Cutoff used: {cutoff_used}\n"
                    "\\begin{align*}\n"
                    f"{label} = & {termUnifiedLegs}\n"
                    "\\end{align*}"
                )

                used_heads.update([head1], [head2])
                used_tails.update([tail1], [tail2])

                equations.append({
                    "label": label,
                    "type": "ensemble",
                    "latex": eq,
                    "group_size": group_size,
                    "source_neurons": list(used_heads),
                    "target_neurons": list(used_tails),
                    "edges": [(head, tail, z) for head, tail, z in group]
                })

            pdb.set_trace()
        #Group size larger than 2 so this is a large ensemble
        else:
            
            # if any("EPG" in h or "EPG" in t or "PEN" in h or "PEN" in t for h, t, _ in group):
            #     print(f"\n🔍 EPG/PEN group detected:")
            #     print(f"  Group key: {key}")
            #     print(f"  Group size: {len(group)}")
            #     for h, t, z in group:
            #         print(f"    {h} → {t}, z={z:.2f}")
            
            roles = identify_ensemble_roles(group, nt_dict, polarity_dict)
            
            # if roles is None and any("EPG" in h or "EPG" in t or "PEN" in h or "PEN" in t for h, t, _ in group):
            #     print(f"🚫 Ensemble roles rejected for group: {key}")
            #     for h, t, z in group:
            #         print(f"  {h} → {t}, z={z:.2f}")

            
            
            
            if roles:
                
                ensemble_count += 1
                
                ensemble_size = len(group)
                label = f"\\mu^{{\\epsilon_{{{roles['post']}^{ensemble_size}}}^{{(P)}}}}"
                term = (
                    f"\\bm{{W}}_{{{roles['post']}^{ensemble_size}}}^{{({roles['polarity']})}}({roles['z_expr']}) "
                    f"\\cdot \\mu_{{{roles['pre']}}}^{{(A)}}"
                )

                eq = (
                    f"% Cutoff used: {cutoff_used}\n"
                    "\\begin{align*}\n"
                    f"{label} = & {term}\n"
                    "\\end{align*}"
                )

                for head, tail, _ in group:
                    used_heads.update([head])
                    used_tails.update([tail])

                equations.append({
                    "label": label,
                    "type": "ensemble",
                    "latex": eq,
                    "group_size": ensemble_size,
                    "source_neurons": list(used_heads),
                    "target_neurons": list(used_tails),
                    "edges": [(head, tail, z) for head, tail, z in group]
                })



    print(f"\n📘 {len(dyad_roles_dict)} Dyad Roles:")
    print(display(JSON(dyad_roles_dict)))




    #########################
    #########################
    #########################    
    #########################
    ######################### 
    # Step 5: Remaining microstates
    
    
    #Debug telemetry dict for microstates
    microstate_dict = {}
    for head, tail, z in xmotif_edges:
        if tail in used_tails or head in used_heads:
            continue

        microstate_dict[f"{head}->{tail}"] = z
    
            
        #microstate_count
        microstate_count += 1
            
        polarity = get_polarity(head, nt_dict, polarity_dict)
        label = f"\\mu_{{{tail}}}^{{(P)}}"
        eq = (
            "\\begin{align*}\n"
            f"{label} = "
            f"\\bm{{W}}_{{{tail} \\leftarrow {head}}}^{{({polarity})}}(z^{{{z:.2f}}}) "
            f"\\cdot \\mu_{{{head}}}^{{(A)}}"
            "\\end{align*}"
        )

        equations.append({
            "label": label,
            "type": "microstate",
            "latex": eq,
            "group_size": 1,
            "source_neurons": [head],
            "target_neurons": [tail],
            "edges": [(head, tail, z)]
        })
        
    
    print(f"\n🔍 Display {len(microstate_dict)} neurons in microstate dict:")
    print(display(JSON(microstate_dict)))
    
    print("\n🔍 Equation Routing Summary:")
    print(f"  Recurrent groups emitted: {recurrent_count}") 
    print(f"  Dyads emitted: {dyad_count}")
    print(f"  Large Ensembles emitted: {ensemble_count}")
    print(f"  Fanin ensemble emitted: {fanin_count}")
    print(f"  Fanout ensemble emitted: {fanout_count}")
    print(f"  Microstates emitted: {microstate_count}")

    return equations

def abbreviate_motif_type(motif_type):
    words = re.split(r'[^a-zA-Z0-9]+', motif_type)
    return ''.join(word[0].upper() for word in words if word)




def build_motif_equations(motif_dict, nt_dict, polarity_dict):
    equations = []

    for motif in motif_dict.values():
        motif_type = motif.get("motifType", "unknown")
        if abbreviate_motif_type(motif_type).upper() != "WC":
            continue  # Only handle WC motifs here

        A = motif["neuron_A"]
        B = motif["neuron_B"]
        z_A = motif.get("Z_A", 1.0)
        z_B = motif.get("Z_B", 1.0)
        polarity_A = get_polarity(A, nt_dict, polarity_dict)
        polarity_B = get_polarity(B, nt_dict, polarity_dict)
        abb_mt = abbreviate_motif_type(motif_type)
        motif_label = f"{abb_mt.upper()}_{{{A},{B}}}"

        eq = (
            "\\begin{align*}\n"
            f"\\mu_{{{motif_label}}}^{{(P)}} = & \\bm{{W}}_{{{motif_label} \\leftarrow {A}}}^{{({polarity_A})}}(z^{{{z_A:.2f}}}) "
            f"\\cdot \\mu_{{{A}}}^{{(A)}} \\\\\n"
            f"+ & \\bm{{W}}_{{{motif_label} \\leftarrow {B}}}^{{({polarity_B})}}(z^{{{z_B:.2f}}}) "
            f"\\cdot \\mu_{{{B}}}^{{(A)}}\n"
            "\\end{align*}"
        )

        
        equations.append({
            "label": motif_label,
            "type": "motif",
            "latex": eq,
            "source_neurons": [A, B],
            "motif_type": abb_mt.upper(),
            "edges": [(A, motif_label, z_A), (B, motif_label, z_B)]
        })


    return equations



def write_latex(equations, filename):
    with open(filename, "w") as f:
        f.write("\\section*{Predictive Coding Equations}\n\n")
        for eq in equations:
            stripped = eq.strip()

            # Check if the block contains any math environment
            if "\\begin{align" in stripped or "\\begin{multline" in stripped or "\\begin{equation" in stripped:
                f.write(stripped + "\n\n")
            elif stripped.startswith("\\section*") or stripped.startswith("\\subsection*"):
                f.write(stripped + "\n\n")
            else:
                f.write("\\begin{equation*}\n")
                f.write(stripped + "\n")
                f.write("\\end{equation*}\n\n")


                             

def main():
    folder = "PC_LatexGen"
    dot_file = os.path.join(folder, "MBON03_prePost_merged_filtered.gv")
    nt_file = os.path.join(folder, "neuronNTdict.json")
    motif_file = os.path.join(folder, "motifs.json")
    

    polarity_dict = {
        "acetylcholine": 1,
        "glutamate": 0,
        "gaba": 0,
        "dopamine": 1,
        "octopamine": 1,
        "serotonin": 0
    }
    

    
    # Load data
    dot_lines, nt_dict, motif_dict = load_files(dot_file, nt_file, motif_file)
    edges = extract_edges(dot_lines)

    # Generate equations
    neuron_eqs = build_neuron_equations(edges, nt_dict, polarity_dict, motif_dict)
    motif_eqs = build_motif_equations(motif_dict, nt_dict, polarity_dict)
    
    grouped_eqs = defaultdict(list)
    for eq in neuron_eqs + motif_eqs:
        grouped_eqs[eq["type"]].append(eq)

    # Define section titles
    section_titles = {
        "motif": "Motif Equations",
        "ensemble": "Ensemble Equations",
        "pooled": "Pooled Equations",
        "fan_in": "Fan-In Equations",
        "fan_out": "Fan-Out Equations",
        "microstate": "Remaining Microstates"
    }

    # Build LaTeX blocks with section headers
    latex_blocks = []
    for eq_type in sorted(grouped_eqs.keys()):
        title = section_titles.get(eq_type, eq_type.title())
        latex_blocks.append(f"\n\\section*{{{title}}}\n")
        latex_blocks.extend(eq["latex"] for eq in grouped_eqs[eq_type])

    # Write LaTeX
    output_path = os.path.join(folder, "predictive_coding_equations.tex")
    write_latex(latex_blocks, output_path)
    print(f"LaTeX file written to: {output_path}")



if __name__ == "__main__":
    main()




Getting data from files:
12 recurrent motifs found


<IPython.core.display.JSON object>

15 Fan-in neurons found


<IPython.core.display.JSON object>

6 Fan-in neurons found


<IPython.core.display.JSON object>

> /tmp/ipykernel_466/1984566609.py(678)build_neuron_equations()
    676 
    677     # Emit Dyad ensemble equations
--> 678     for key, group in abstracted_groups.items():
    679 
    680         group_size = len(group)



ipdb>  q
